# GCM BA - 8. Ad hoc test: the low end of the precipitation distributions, to choose the trace threshold

**Ad hoc test, outside the chain**: it reads ERA5 (0.25 deg, and on the EC-Earth3 grid) and EC-Earth3 historical, and
writes only in `GCM_BA/8_test_trace/` (EOS: cache) and `outputs/` next to it.

**Question.** The supervisor's pipeline (ClimDown) replaces the precipitation below a *trace* threshold by a uniform
random value in (0, trace) before the QDM (Cannon jitter), in the model and in the reference: below the trace a value
is treated as a dry day, above it as rain whose quantiles are mapped. The pipeline uses trace = 0.005 mm/day, set for
CERRA; F1 and ISIMIP3BASD use 0.1 mm/day. With ERA5 as reference the right value may differ. Following the supervisor,
the trace is chosen by comparing the low end of the daily precipitation distributions of the model and of the
reference: it should lie above the range where one of them has a mass of tiny values that the other does not
(numerical drizzle of the model, residues of the ERA5 accumulations or of the regridding), since the QDM would map that
noise onto real rain or the reverse, with very large ratios.

**Datasets** (`YEARS`, domain pixels: European land of LISFLOOD and the sea along it, area-weighted):
- ERA5 at 0.25 deg: the reference of the pipeline;
- ERA5 on the EC-Earth3 grid (CDO, as F1): the effect of the regridding on the small values;
- EC-Earth3 historical, raw (0.7 deg);
- EC-Earth3 after a climate imprint at 0.25 deg, as the first block of the pipeline (ClimDown principle): the daily
  ratio of the model to its own monthly climatology, interpolated bilinearly to 0.25 deg, times the ERA5 monthly
  climatology. This is what enters the QDM; the bilinear interpolation of the daily ratios between a dry and a wet
  cell creates small values (the drizzle that the pipeline's `fix_drizzle` targets). Approximation: climatologies over
  `YEARS`, no outlier filter.

**Outputs.** (1) Share of the days with pr < t for t from 0.0001 to 1 mm/day (exact zeros and negative values apart),
per season (DJF, JJA, year) and dataset; (2) the curves of that share against t; (3) maps of the share of the days with
0.005 ≤ pr < 0.1 mm/day: the range between the two candidate traces.

**Reading.** Compare the mass between 0.005 and 0.1 mm/day in ERA5 0.25 deg and in the model after the climate
imprint: if it is large in one and small in the other, a trace of 0.005 lets that noise into the QDM, and the trace
should be raised above it. The curves should become parallel (same shape, real rain in both) above the right trace.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, time, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
CAL_PERIOD = ("1981", "2014")                          # calibration period (ERA5 / EC-Earth3 historical)
YEARS = tuple(os.environ.get("TRACE_YEARS", "2005,2014").split(","))   # 10 years of it are enough for the shares
MARGIN = 1.0                                           # degrees around the EFAS domain, as alternative_solution_BA_RCM/1
REGION = {k: (bu.EFAS_BOX[k][0] - MARGIN, bu.EFAS_BOX[k][1] + MARGIN) for k in ("lat", "lon")}
COARSE_REGION = {k: (REGION[k][0] - 1.0, REGION[k][1] + 1.0) for k in ("lat", "lon")}   # + 1 coarse cell, for the interpolation
THRESHOLDS = [0.0001, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]   # mm/day; 0.005: pipeline, 0.1: F1 / ISIMIP3BASD
EDGES = np.unique(np.r_[0.0, np.logspace(-6, 3, 91), THRESHOLDS])   # bins of the distributions (mm/day)
MID = (0.005, 0.1)                                     # range between the two candidate traces (maps)
SEASONS = {"DJF": (12, 1, 2), "JJA": (6, 7, 8), "year": tuple(range(1, 13))}
OVERWRITE = False

CACHE = bu.gcm_ba_dir(cfg, "8_test_trace", f"cache_{YEARS[0]}-{YEARS[1]}")   # the only folder written on EOS
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
assert CAL_PERIOD[0] <= YEARS[0] <= YEARS[1] <= CAL_PERIOD[1], "YEARS must lie in the calibration period"
print(f"years {YEARS[0]}-{YEARS[1]} | region {REGION} | thresholds {THRESHOLDS} mm/day | cache {CACHE}")

## 1. Inputs and the climate imprint

ERA5 at 0.25 deg from the extract of `alternative_solution_BA_RCM/1` when it exists (EFAS domain + 1 deg, 1981-2014),
otherwise from the global daily files. Climate imprint: monthly climatologies over `YEARS`; ratio = pr / climatology of
the model (0 where the climatology is 0), interpolated bilinearly from the EC-Earth3 grid to the ERA5 grid, times the
ERA5 climatology.

In [ ]:
E_FILE = bu.alt_dir(cfg, "0_inputs") / f"era5_025_efas{MARGIN:g}_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}.nc"
if E_FILE.exists():
    era5_fine = bu.subset_time(bu.read_processed(E_FILE, "pr"), *YEARS)
else:
    era5_fine = bu.subset_time(bu.crop_bbox(bu.read_raw(cfg, "era5", "pr"), REGION), *YEARS)
era5_coarse = bu.gcm_ba_input(cfg, "era5", "pr", YEARS, REGION)
gcm = bu.gcm_ba_input(cfg, "gcm_historical", "pr", YEARS, COARSE_REGION)
gcm_in = bu.crop_bbox(gcm, REGION)

clim_gcm = gcm.groupby("time.month").mean("time").load()
clim_era5 = era5_fine.groupby("time.month").mean("time").load()


def imprint(year):
    # climate imprint of one year of EC-Earth3 onto the ERA5 0.25 deg grid (ratio to the model climatology)
    g = gcm.sel(time=str(year)).load()
    c = clim_gcm.sel(month=g.time.dt.month).drop_vars("month")
    ratio = xr.where(c > 0, g / c.where(c > 0), 0.0)
    fine = ratio.interp(lat=era5_fine.lat, lon=era5_fine.lon, method="linear")
    return fine * clim_era5.sel(month=fine.time.dt.month).drop_vars("month")


SOURCES = {"ERA5 0.25 deg": lambda y: era5_fine.sel(time=str(y)).load(),
           "ERA5 on the EC-Earth3 grid": lambda y: era5_coarse.sel(time=str(y)).load(),
           "EC-Earth3 raw": lambda y: gcm_in.sel(time=str(y)).load(),
           "EC-Earth3 climate imprint 0.25 deg": imprint}
MASK = {"fine": bu.load_domain_mask(cfg, "era5", like=era5_fine.isel(time=0)).values,
        "coarse": bu.load_domain_mask(cfg, "ecearth3", like=era5_coarse.isel(time=0)).values}
GRID_OF = {"ERA5 0.25 deg": "fine", "ERA5 on the EC-Earth3 grid": "coarse", "EC-Earth3 raw": "coarse",
           "EC-Earth3 climate imprint 0.25 deg": "fine"}
for k in SOURCES:
    print(f"{k:36s} {MASK[GRID_OF[k]].sum()} domain pixels")

## 2. Distributions of the daily values (area-weighted counts per bin, per season), year by year

Categories: negative values, exact zeros, then the bins of `EDGES`. Also per pixel: the share of the days in `MID`
(all seasons), and the smallest positive value of each dataset (the resolution of the stored values).

In [ ]:
NCAT = EDGES.size + 2


def categories(v):
    # 0: negative, 1: exact zero, k >= 2: [EDGES[k - 2], EDGES[k - 1]) (k = 2: (0, EDGES[1])), last: >= EDGES[-1]
    c = np.searchsorted(EDGES, v, side="right") + 1
    c[v == 0] = 1
    c[v < 0] = 0
    return c


def distribution(name):
    f = CACHE / f"dist_{name.replace(' ', '_').replace('.', '')}.npz"
    if f.exists() and not OVERWRITE:
        z = np.load(f)
        return {s: z[s] for s in SEASONS}, z["mid"], z["days"], float(z["minpos"])
    m = MASK[GRID_OF[name]]
    acc = {s: np.zeros(NCAT) for s in SEASONS}
    mid, days, minpos = np.zeros(m.shape), 0, np.inf
    t0 = time.time()
    for y in range(int(YEARS[0]), int(YEARS[1]) + 1):
        da = SOURCES[name](y)
        x = da.values
        mon = da.time.dt.month.values
        w = np.cos(np.deg2rad(da.lat.values))[:, None] * np.ones(m.shape)
        mid += np.nansum((x >= MID[0]) & (x < MID[1]), axis=0)
        days += x.shape[0]
        pos = x[np.isfinite(x) & (x > 0)]
        minpos = min(minpos, float(pos.min())) if pos.size else minpos
        for s, months in SEASONS.items():
            v = x[np.isin(mon, months)][:, m]
            ww = np.broadcast_to(w[m], v.shape)
            ok = np.isfinite(v)
            acc[s] += np.bincount(categories(v[ok]), weights=ww[ok], minlength=NCAT)
    mid = np.where(m, mid / max(days, 1), np.nan)
    np.savez(f, mid=mid, days=days, minpos=minpos, **acc)
    print(f"{name}: {(time.time() - t0) / 60:.1f} min", flush=True)
    return acc, mid, days, minpos


DIST, MIDMAP, MINPOS = {}, {}, {}
for name in SOURCES:
    DIST[name], MIDMAP[name], _, MINPOS[name] = distribution(name)
print("smallest positive value (mm/day):", {k: f"{v:.2e}" for k, v in MINPOS.items()})

## 3. Share of the days below each threshold, curves, maps

In [ ]:
def below(acc):
    # share of the days with pr < EDGES[i] (index i of EDGES), negative values and zeros included
    cum = np.cumsum(acc) / acc.sum()
    return np.r_[cum[1], cum[2:EDGES.size + 1]]          # EDGES[0] = 0: the share <= 0 (negatives + zeros)


rows = []
for name, d in DIST.items():
    for s, acc in d.items():
        b = below(acc)
        r = {"dataset": name, "season": s, "negative (%)": 100 * acc[0] / acc.sum(), "exact zero (%)": 100 * acc[1] / acc.sum()}
        for t in THRESHOLDS:
            r[f"< {t:g}"] = 100 * b[np.searchsorted(EDGES, t)]
        r[f"{MID[0]:g} to {MID[1]:g} (%)"] = r[f"< {MID[1]:g}"] - r[f"< {MID[0]:g}"]
        rows.append(r)
TAB = pd.DataFrame(rows).set_index(["season", "dataset"]).sort_index()
TAB.to_csv(OUT / f"pr_low_tail_{YEARS[0]}-{YEARS[1]}.csv")
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print("share of the days (%) below each threshold (mm/day), domain, area-weighted")
    display(TAB.round(2))

COL = {"ERA5 0.25 deg": "k", "ERA5 on the EC-Earth3 grid": "0.55", "EC-Earth3 raw": "#d95f02",
       "EC-Earth3 climate imprint 0.25 deg": "#7570b3"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)
for ax, s in zip(axes, SEASONS):
    for name, d in DIST.items():
        ax.plot(EDGES[1:], 100 * below(d[s])[1:], color=COL[name], lw=1.6, label=name)
    for t, lab in ((0.005, "0.005 (pipeline)"), (0.1, "0.1 (F1, ISIMIP3BASD)")):
        ax.axvline(t, color="grey", ls=":", lw=1); ax.text(t, 2, lab, rotation=90, fontsize=8, va="bottom", ha="right")
    ax.set_xscale("log"); ax.set_xlim(1e-5, 10); ax.set_ylim(0, 100); ax.grid(alpha=0.3)
    ax.set_title(s); ax.set_xlabel("threshold t (mm/day)")
axes[0].set_ylabel("share of the days with pr < t (%)"); axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle(f"Low end of the daily precipitation distributions, domain, {YEARS[0]}-{YEARS[1]}", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / f"pr_low_tail_curves_{YEARS[0]}-{YEARS[1]}.png", dpi=110, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))
for ax, name in zip(axes, SOURCES):
    g = era5_fine if GRID_OF[name] == "fine" else era5_coarse
    im = ax.pcolormesh(g.lon, g.lat, 100 * MIDMAP[name], vmin=0, vmax=40, cmap="viridis", shading="nearest")
    ax.set_title(f"{name}\n{MID[0]:g} <= pr < {MID[1]:g} mm/day (% of days)", fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.8, label="% of days")
fig.savefig(OUT / f"pr_low_tail_maps_{YEARS[0]}-{YEARS[1]}.png", dpi=90, bbox_inches="tight"); plt.show()

## 4. Reading

To be written after the run on BDAP: (1) the mass between 0.005 and 0.1 mm/day in ERA5 0.25 deg and in EC-Earth3
after the climate imprint, per season; (2) where the curves become parallel; (3) the trace proposed for the pipeline
with ERA5 (and whether the values below it are set back to zero after the QDM: the line is commented out in
`applyfactors.R`).